# Sham — CPU Training Track (self-schedulable, no GPU quota)

**Why this notebook exists (owner, 2026-09-20):** the main training notebook is capped by Kaggle's real, documented **30 GPU-hours/week** quota. CPU compute is **not** subject to that cap — a CPU-only notebook can run far more often, accumulating real training hours the GPU track structurally can't. It trains much more slowly per step (dense transformer training on CPU vs. a T4 GPU is a real, large slowdown — expect roughly one to two orders of magnitude fewer steps/second, not a small difference), which is exactly why this is framed as "slow but continuous" rather than a GPU replacement.

**Because this notebook has NO accelerator, Kaggle's own native "Schedule this notebook" feature accepts it directly** (only GPU-attached notebooks are refused) — no separate orchestrator notebook is needed for this one, unlike the GPU training notebook.

**Design decision — a SEPARATE checkpoint lineage, not a shared one:** this track publishes its own checkpoints to its own Kaggle Dataset (`sham-cpu-track-checkpoint`), never back into the GPU track's `nova-small-checkpoint` dataset. Two independent processes training the SAME weights on different schedules, unaware of each other, would race: whichever happens to publish last "wins" regardless of whether its own steps were actually better, which could silently regress the model. Keeping the lineages separate avoids that entirely.

**One-time fork from the GPU track (manual, on purpose):** to start this track from real, already-trained weights instead of a fresh random init, attach the GPU track's `nova-small-checkpoint` dataset as an Input for the FIRST run of this notebook only, then remove it and only ever keep this track's own `sham-cpu-track-checkpoint` attached from the second run onward. This mirrors the same "you manage which Input datasets are attached" convention the main training notebook already uses for its own resume step — nothing new, just applied once, deliberately, instead of automatically.

**One-time setup checklist:**
1. Accelerator: **None**. Internet: **On**.
2. Secrets (Add-ons): `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY` — same three already used by the GPU notebook.
3. Create an empty Kaggle Dataset named `sham-cpu-track-checkpoint` once (Kaggle → New Dataset → upload any placeholder file) so the publish step below has something to version into.
4. First run only: Add Input → the GPU track's `nova-small-checkpoint` dataset (for the one-time fork).
5. Save, then use Kaggle's own **"Schedule this notebook"** — as often as you like; there is no 30h/week cap to plan around here.

### 1) Clone the real code from GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("Sham code ready at:", CODE_DIR)

In [ ]:
try:
    import tokenizers
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
print("tokenizers ready.")

### 2) Real Arabic text data (same streaming source as the GPU track)

In [ ]:
from data_acquisition import stream_hf_text_corpus

MAX_DOCUMENTS = 5_000  # smaller than the GPU track on purpose -- a CPU session runs far fewer real steps per run, so it doesn't need as large a shard to stay fed

corpus_dir = "/kaggle/working/corpus/wikipedia_ar"
corpus_files = stream_hf_text_corpus(
    dataset_name="wikimedia/wikipedia",
    config_name="20231101.ar",
    text_field="text",
    output_dir=corpus_dir,
    max_documents=MAX_DOCUMENTS,
)
print(f"shard files: {len(corpus_files)}")

### 3) Tokenizer — MUST match the GPU track's vocab for the one-time fork to load correctly

If resuming from this track's own prior checkpoint, its own saved tokenizer is reused. On the very first run (forking from the GPU track), reuse the GPU track's own saved tokenizer file (attached via its dataset Input) instead of training a fresh one -- a mismatched vocabulary would silently corrupt every forked weight's meaning.

In [ ]:
from pathlib import Path
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from model import TEXT_VOCAB_SIZE

previous_tokenizer_files = list(Path("/kaggle/input").rglob("*tokenizer*.json"))
if previous_tokenizer_files:
    tokenizer_path = previous_tokenizer_files[0]
    tokenizer = ShamTextTokenizer.load(str(tokenizer_path))
    print(f"reused an existing tokenizer from a previous session/fork: {tokenizer_path}")
else:
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    print("trained a fresh tokenizer (no previous one found -- this should only happen on a true first run).")
tokenizer.save("/kaggle/working/sham_cpu_tokenizer.json")

### 4) Build training windows

In [ ]:
import torch
from dataset import TextSequenceDataset

SEQ_LEN = 512  # shorter than the GPU track's 1024 -- CPU attention cost grows with seq_len^2, and this track's whole point is real completed steps, not a long context per step
text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"real training windows: {len(text_dataset):,}")

### 5) Model size -- MUST match the GPU track's "starter" config exactly for the one-time fork's weights to load

In [ ]:
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE

model_cfg = ShamSmallConfig(
    vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
    mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
)
model = ShamSmall(model_cfg)
print(f"model: {model.count_parameters():,} real parameters.")

device = "cpu"  # this track is CPU-only by design; no accelerator is ever attached to this notebook
print(f"device: {device}")

### 6) Resume -- from THIS track's own prior checkpoint, or the GPU track's (one-time fork only)

Looks across every attached Input, exactly like the GPU notebook's own resume cell -- the safety here comes from the setup checklist above (only the GPU dataset is attached on the very first run, then removed), not from any automatic lineage detection in this cell.

In [ ]:
from checkpoint import load_checkpoint

start_step = 0
resume_optimizer = None

previous_checkpoints = sorted(
    Path("/kaggle/input").rglob("step_*.pt"),
    key=lambda p: int(p.stem.split("_")[1]),
)
if previous_checkpoints:
    last_ckpt = previous_checkpoints[-1]
    model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
    from train import build_optimizer
    resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
    load_checkpoint(last_ckpt, map_location=device, load_optimizer_into=resume_optimizer)
    print(f"resumed from: {last_ckpt} (step {start_step:,})")
else:
    print("no previous checkpoint found -- starting from scratch (expected only on a true first run).")

### 7) Measure real CPU speed, then train for a bounded wall-clock budget

In [ ]:
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

CALIBRATION_STEPS = 10
calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, CALIBRATION_STEPS * 2 * 4), 2)
][: CALIBRATION_STEPS * 4]
assert calib_batches, "not enough data to calibrate -- increase MAX_DOCUMENTS above."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

t0 = time.time()
steps_done = 0
for batch in calib_batches:
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed

# Not necessarily the same limit as GPU sessions -- adjust if your own
# observed CPU session length on Kaggle differs from this conservative default.
MAX_TRAINING_HOURS = 8.5
realistic_steps_for_session = max(int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85), 20)

print(f"real measured speed: {steps_per_second:.4f} steps/sec on {device}")
print(f"realistic steps for this session (with safety margin): {realistic_steps_for_session:,}")

### 8) Train

In [ ]:
TOTAL_STEPS = realistic_steps_for_session
num_windows = len(text_dataset) - (len(text_dataset) % 4)

def _batch_iterator():
    while True:
        for b in range(0, num_windows, 4):
            yield torch.stack([text_dataset[i] for i in range(b, b + 4)])

import itertools
batches = itertools.islice(_batch_iterator(), TOTAL_STEPS)

train_cfg = TrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    grad_accum_steps=4,
    lr=3e-4,
    warmup_steps=max(20, TOTAL_STEPS // 100),
    total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=100,
    log_every=10,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)

from train import train
loss_history = train(
    model, batches, train_cfg, device=device,
    start_step=start_step, resume_optimizer=resume_optimizer or _calib_optimizer,
)
print(f"\nreal steps this session: {len(loss_history):,}")
if loss_history:
    print(f"first 10 avg loss: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
    print(f"last 10 avg loss: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")

### 9) Save + publish this track's own checkpoint (its own dataset, never the GPU track's)

In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final.pt", model, final_step)
print(f"final checkpoint saved locally at step {final_step:,}.")

In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-cpu-track-checkpoint"  # this track's OWN dataset -- never nova-small-checkpoint

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_cpu_tokenizer.json", upload_dir / "sham_cpu_tokenizer.json")

metadata = {"title": "sham-cpu-track-checkpoint", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

result = subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"cpu-track auto-update at step {final_step:,}", "-r", "skip"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("WARNING: failed to publish -- the checkpoint is still safe in this session's own Output. "
          "Check that the sham-cpu-track-checkpoint dataset exists and KAGGLE_USERNAME/KAGGLE_KEY are correct.")
    print(result.stderr)
else:
    print(f"published step {final_step:,} to {DATASET_SLUG} -- the next scheduled run will pick it up automatically.")